# Montagem do dataset_final_treino_v3

**Fase 6 — Consolidação final dos datasets para treino V3**

Fontes:
- `V2_PROPRIA`: dataset_final_treino_v2_balanced (GFC + RSS próprios)
- `FAKETRUEBR`: faketruebr_curated_full (dataset acadêmico)
- `FAKEBR_TEXT_NORMALIZED`: fakebr_curated_text_normalized (dataset acadêmico, texto normalizado por tamanho)

**Regras permanentes:** não alterar V1/V2, não treinar modelo, não sobrescrever CSVs existentes.

In [ ]:
import pandas as pd
import numpy as np
import re
import unicodedata
from datetime import datetime
from pathlib import Path

SEED = 42
np.random.seed(SEED)

# --- Detecção robusta da raiz do projeto ---
def _find_project_root() -> Path:
    cwd = Path.cwd()
    if (cwd / 'dados').is_dir():
        return cwd
    if (cwd.parent / 'dados').is_dir():
        return cwd.parent
    raise FileNotFoundError(
        f"Pasta 'dados' não encontrada em '{cwd}' nem em '{cwd.parent}'.\n"
        "Execute o notebook a partir da raiz do projeto ou de src/."
    )

PROJECT_ROOT = _find_project_root()

# --- Caminhos ---
DIR_FINAL       = PROJECT_ROOT / 'dados' / 'dataset_unificado' / 'final'
DIR_FAKETRUEBR  = PROJECT_ROOT / 'dados' / 'pipeline_datasets_academicos' / 'curated' / 'faketruebr'
DIR_FAKEBR      = PROJECT_ROOT / 'dados' / 'pipeline_datasets_academicos' / 'curated' / 'fakebr'
DIR_ACAD_FINAL  = PROJECT_ROOT / 'dados' / 'pipeline_datasets_academicos' / 'final'

DIR_FINAL.mkdir(parents=True, exist_ok=True)

print(f'PROJECT_ROOT : {PROJECT_ROOT}')
print(f'DIR_FINAL    : {DIR_FINAL}')
print(f'DIR_FAKETRUEBR: {DIR_FAKETRUEBR}')
print(f'DIR_FAKEBR   : {DIR_FAKEBR}')
print(f'DIR_ACAD_FINAL: {DIR_ACAD_FINAL}')

In [ ]:
# --- Localização automática dos arquivos mais recentes ---
def _mais_recente(diretorio: Path, padrao: str) -> Path:
    """Retorna o arquivo mais recente que corresponde ao padrão glob."""
    candidatos = sorted(diretorio.glob(padrao))
    if not candidatos:
        raise FileNotFoundError(f'Nenhum arquivo '{padrao}' em {diretorio}')
    return candidatos[-1]  # ordem lexicográfica = ordem cronológica pelo timestamp no nome

ARQUIVO_V2_BALANCED   = _mais_recente(DIR_FINAL,      'dataset_final_treino_v2_balanced_*.csv')
ARQUIVO_FAKETRUEBR    = _mais_recente(DIR_FAKETRUEBR, 'faketruebr_curated_full_*.csv')
ARQUIVO_FAKEBR_NORM   = _mais_recente(DIR_FAKEBR,     'fakebr_curated_text_normalized_*.csv')

# Overlap reports (opcionais)
try:
    ARQUIVO_OVERLAP_RPT   = _mais_recente(DIR_ACAD_FINAL, 'overlap_report_*.csv')
except FileNotFoundError:
    ARQUIVO_OVERLAP_RPT = None

try:
    ARQUIVO_NEAR_DUPS     = _mais_recente(DIR_ACAD_FINAL, 'overlap_near_duplicates_*.csv')
except FileNotFoundError:
    ARQUIVO_NEAR_DUPS = None

SEP = '=' * 70
print(SEP)
print('ARQUIVOS SELECIONADOS')
print(SEP)
print(f'V2 balanced      : {ARQUIVO_V2_BALANCED.name}')
print(f'FakeTrueBR       : {ARQUIVO_FAKETRUEBR.name}')
print(f'FakeBR TextNorm  : {ARQUIVO_FAKEBR_NORM.name}')
print(f'Overlap report   : {ARQUIVO_OVERLAP_RPT.name if ARQUIVO_OVERLAP_RPT else "NÃO ENCONTRADO"}')
print(f'Near duplicates  : {ARQUIVO_NEAR_DUPS.name if ARQUIVO_NEAR_DUPS else "NÃO ENCONTRADO"}')

In [ ]:
# --- Carregar datasets ---
df_v2   = pd.read_csv(ARQUIVO_V2_BALANCED,  encoding='utf-8')
df_ftbr = pd.read_csv(ARQUIVO_FAKETRUEBR,   encoding='utf-8')
df_fbr  = pd.read_csv(ARQUIVO_FAKEBR_NORM,  encoding='utf-8')

# Overlap reports
df_overlap_rpt  = pd.read_csv(ARQUIVO_OVERLAP_RPT)  if ARQUIVO_OVERLAP_RPT  else pd.DataFrame()
df_near_dups    = pd.read_csv(ARQUIVO_NEAR_DUPS)     if ARQUIVO_NEAR_DUPS    else pd.DataFrame()

print(SEP)
print('TAMANHOS CARREGADOS')
print(SEP)
print(f'V2 balanced   : {len(df_v2):,} registros | colunas: {list(df_v2.columns)}')
print(f'FakeTrueBR    : {len(df_ftbr):,} registros | colunas: {list(df_ftbr.columns)}')
print(f'FakeBR TextNorm: {len(df_fbr):,} registros | colunas: {list(df_fbr.columns)}')
print()
if len(df_near_dups) > 0:
    print(f'Near-duplicates (FakeTrueBR × FakeBR): {len(df_near_dups)} pares')
    print(df_near_dups.to_string())

In [ ]:
# --- Funções auxiliares ---

def _classificar_faixa(n: float) -> str:
    if pd.isna(n) or n < 0:
        return 'desconhecido'
    if n < 500:
        return 'curto'
    elif n < 2000:
        return 'medio'
    elif n < 5000:
        return 'longo'
    return 'muito_longo'

def _normalizar_texto(texto: str) -> str:
    """Remove acentos, lowercase, colapsa espaços — usado só para deduplicação."""
    if not isinstance(texto, str):
        return ''
    t = unicodedata.normalize('NFKD', texto)
    t = ''.join(c for c in t if not unicodedata.combining(c))
    t = t.lower()
    t = re.sub(r'\s+', ' ', t).strip()
    return t

def _normalizar_url(url) -> str:
    """Normalização mínima de URL para deduplicação."""
    if not isinstance(url, str) or url.strip() == '':
        return ''
    u = url.strip().rstrip('/').lower()
    u = re.sub(r'^https?://', '', u)
    u = re.sub(r'^www\.', '', u)
    return u

# Colunas do schema final V3
COLUNAS_V3 = [
    'id_registro',
    'texto_principal',
    'texto_principal_modelo',
    'label',
    'label_detalhe',
    'pipeline_origem',
    'portal_origem',
    'origem_texto',
    'origem_qualidade',
    'tamanho_chars',
    'tamanho_chars_modelo',
    'data_publicacao',
    'url_origem',
    'fonte_dataset',
    'referencia_dataset',
    'dataset_origem',
    'arquivo_origem',
    'faixa_tamanho',
    'faixa_tamanho_modelo',
]

print('Schema V3 definido com', len(COLUNAS_V3), 'colunas:')
for c in COLUNAS_V3:
    print(f'  - {c}')

In [ ]:
# --- Padronizar V2 ---
df_v2_std = df_v2.copy()

# V2 não tem texto_principal_modelo: usar texto_principal
df_v2_std['texto_principal_modelo'] = df_v2_std['texto_principal']
df_v2_std['tamanho_chars_modelo']   = df_v2_std['texto_principal_modelo'].str.len()
df_v2_std['faixa_tamanho']          = df_v2_std['tamanho_chars'].apply(_classificar_faixa)
df_v2_std['faixa_tamanho_modelo']   = df_v2_std['tamanho_chars_modelo'].apply(_classificar_faixa)
df_v2_std['fonte_dataset']          = pd.NA
df_v2_std['referencia_dataset']     = pd.NA
df_v2_std['dataset_origem']         = 'V2_PROPRIA'
df_v2_std['arquivo_origem']         = ARQUIVO_V2_BALANCED.name

df_v2_std = df_v2_std[COLUNAS_V3]

print(f'V2 padronizado: {len(df_v2_std)} registros')
print(df_v2_std[['id_registro', 'dataset_origem', 'label', 'tamanho_chars_modelo']].head(3).to_string())

In [ ]:
# --- Padronizar FakeTrueBR ---
df_ftbr_std = df_ftbr.copy()

# FakeTrueBR não tem texto_principal_modelo: usar texto_principal
df_ftbr_std['texto_principal_modelo'] = df_ftbr_std['texto_principal']
df_ftbr_std['tamanho_chars_modelo']   = df_ftbr_std['texto_principal_modelo'].str.len()
if 'faixa_tamanho' not in df_ftbr_std.columns:
    df_ftbr_std['faixa_tamanho'] = df_ftbr_std['tamanho_chars'].apply(_classificar_faixa)
df_ftbr_std['faixa_tamanho_modelo']   = df_ftbr_std['tamanho_chars_modelo'].apply(_classificar_faixa)
df_ftbr_std['dataset_origem']         = 'FAKETRUEBR'
df_ftbr_std['arquivo_origem']         = ARQUIVO_FAKETRUEBR.name

df_ftbr_std = df_ftbr_std[COLUNAS_V3]

print(f'FakeTrueBR padronizado: {len(df_ftbr_std)} registros')
print(df_ftbr_std[['id_registro', 'dataset_origem', 'label', 'tamanho_chars_modelo']].head(3).to_string())

In [ ]:
# --- Padronizar FakeBR text_normalized ---
df_fbr_std = df_fbr.copy()

# FakeBR text_normalized já tem texto_principal_modelo e tamanho_chars_modelo
if 'faixa_tamanho' not in df_fbr_std.columns:
    df_fbr_std['faixa_tamanho'] = df_fbr_std['tamanho_chars'].apply(_classificar_faixa)
if 'faixa_tamanho_modelo' not in df_fbr_std.columns:
    df_fbr_std['faixa_tamanho_modelo'] = df_fbr_std['tamanho_chars_modelo'].apply(_classificar_faixa)
df_fbr_std['dataset_origem']  = 'FAKEBR_TEXT_NORMALIZED'
df_fbr_std['arquivo_origem']  = ARQUIVO_FAKEBR_NORM.name

df_fbr_std = df_fbr_std[COLUNAS_V3]

print(f'FakeBR TextNorm padronizado: {len(df_fbr_std)} registros')
print(df_fbr_std[['id_registro', 'dataset_origem', 'label', 'tamanho_chars_modelo']].head(3).to_string())

In [ ]:
# --- Concatenar e registrar contagens iniciais ---
n_inicial_v2   = len(df_v2_std)
n_inicial_ftbr = len(df_ftbr_std)
n_inicial_fbr  = len(df_fbr_std)
n_inicial_total = n_inicial_v2 + n_inicial_ftbr + n_inicial_fbr

df_merged = pd.concat([df_v2_std, df_ftbr_std, df_fbr_std], ignore_index=True)

print(SEP)
print('CONTAGENS INICIAIS APÓS CONCATENAÇÃO')
print(SEP)
print(f'V2_PROPRIA             : {n_inicial_v2:,}')
print(f'FAKETRUEBR             : {n_inicial_ftbr:,}')
print(f'FAKEBR_TEXT_NORMALIZED : {n_inicial_fbr:,}')
print(f'Total                  : {n_inicial_total:,}')
print()
print('Distribuição por label:')
print(df_merged['label'].value_counts().rename({0: 'label=0 (fake)', 1: 'label=1 (real)'}).to_string())

In [ ]:
# --- SEÇÃO 5: Limpeza e validação ---
print(SEP)
print('SEÇÃO 5 — LIMPEZA E VALIDAÇÃO')
print(SEP)

df_clean = df_merged.copy()

# 5.1 Garantir texto_principal_modelo não nulo
mask_sem_tpm = df_clean['texto_principal_modelo'].isna() | (df_clean['texto_principal_modelo'].astype(str).isin(['', 'nan', 'None', 'NaN']))
# Fallback: usar texto_principal
df_clean.loc[mask_sem_tpm, 'texto_principal_modelo'] = df_clean.loc[mask_sem_tpm, 'texto_principal']

# Agora remover os que ainda não têm texto_principal_modelo
mask_sem_tpm2 = df_clean['texto_principal_modelo'].isna() | (df_clean['texto_principal_modelo'].astype(str).isin(['', 'nan', 'None', 'NaN']))
n_sem_tpm = int(mask_sem_tpm2.sum())
df_clean = df_clean[~mask_sem_tpm2].copy()

# 5.2 Remover sem label
mask_sem_label = ~df_clean['label'].isin([0, 1])
n_sem_label = int(mask_sem_label.sum())
df_clean = df_clean[~mask_sem_label].copy()

# 5.3 Normalizar espaços em texto_principal_modelo
df_clean['texto_principal_modelo'] = (
    df_clean['texto_principal_modelo']
    .astype(str)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

# 5.4 Recalcular tamanho_chars_modelo
df_clean['tamanho_chars_modelo'] = df_clean['texto_principal_modelo'].str.len()
df_clean['faixa_tamanho_modelo'] = df_clean['tamanho_chars_modelo'].apply(_classificar_faixa)

# 5.5 Criar texto_norm (apenas para deduplicação, não salvo)
df_clean['_texto_norm'] = df_clean['texto_principal_modelo'].apply(_normalizar_texto)

# 5.6 Criar url_norm (apenas para deduplicação, não salvo)
df_clean['_url_norm'] = df_clean['url_origem'].apply(_normalizar_url)

print(f'Removidos sem texto_principal_modelo : {n_sem_tpm}')
print(f'Removidos sem label válido           : {n_sem_label}')
print(f'Total após limpeza básica            : {len(df_clean):,}')

In [ ]:
# --- SEÇÃO 6: Remoção de duplicatas e overlaps ---
print(SEP)
print('SEÇÃO 6 — REMOÇÃO DE DUPLICATAS E OVERLAPS')
print(SEP)

df_dedup = df_clean.copy()

# Prioridade de deduplicação: V2_PROPRIA > FAKEBR_TEXT_NORMALIZED > FAKETRUEBR
PRIORIDADE = {'V2_PROPRIA': 0, 'FAKEBR_TEXT_NORMALIZED': 1, 'FAKETRUEBR': 2}
df_dedup['_prioridade'] = df_dedup['dataset_origem'].map(PRIORIDADE).fillna(99).astype(int)
df_dedup = df_dedup.sort_values('_prioridade').reset_index(drop=True)

n_antes_dedup = len(df_dedup)

# 6.1 Remover duplicatas exatas por texto_norm
dup_texto = df_dedup.duplicated(subset=['_texto_norm'], keep='first')
n_dup_texto = int(dup_texto.sum())
df_dedup = df_dedup[~dup_texto].copy()
print(f'Removidos por texto_norm duplicado   : {n_dup_texto}')

# 6.2 Remover duplicatas por url_norm (apenas URLs não vazias)
df_dedup = df_dedup.reset_index(drop=True)
mask_url_valida = df_dedup['_url_norm'] != ''
dup_url_sub = df_dedup.loc[mask_url_valida, '_url_norm'].duplicated(keep='first')
dup_url_idx = set(dup_url_sub[dup_url_sub].index.tolist())
n_dup_url = len(dup_url_idx)
df_dedup = df_dedup.loc[~df_dedup.index.isin(dup_url_idx)].copy()
print(f'Removidos por url_norm duplicada     : {n_dup_url}')

# 6.3 Near-duplicates (FakeTrueBR × FakeBR): remover entradas FAKETRUEBR
n_near_dup_removidos = 0
if len(df_near_dups) > 0 and 'idx_FakeTrueBR' in df_near_dups.columns:
    ids_near_dup_ftbr = set(df_near_dups['idx_FakeTrueBR'].astype(str).tolist())
    mask_near_dup = (
        (df_dedup['dataset_origem'] == 'FAKETRUEBR') &
        (df_dedup['id_registro'].astype(str).isin(ids_near_dup_ftbr))
    )
    n_near_dup_removidos = int(mask_near_dup.sum())
    df_dedup = df_dedup[~mask_near_dup].copy()
print(f'Removidos por near-duplicate (FTBR)  : {n_near_dup_removidos}')

# 6.4 Detectar conflitos de label (mesmo texto_norm, labels diferentes)
# (após dedup, só sobra conflito se o mesmo texto aparecer com labels 0 e 1)
conflitos = df_dedup.groupby('_texto_norm')['label'].nunique()
ids_conflito = set(conflitos[conflitos > 1].index.tolist())
mask_conflito = df_dedup['_texto_norm'].isin(ids_conflito)
n_conflito = int(mask_conflito.sum())
df_conflitos_log = df_dedup[mask_conflito][COLUNAS_V3].copy()
df_dedup = df_dedup[~mask_conflito].copy()
print(f'Removidos por conflito de label      : {n_conflito}')

# Limpar colunas auxiliares
df_dedup = df_dedup.drop(columns=['_texto_norm', '_url_norm', '_prioridade'])

n_apos_dedup = len(df_dedup)
print()
print(f'Total antes de deduplicação          : {n_antes_dedup:,}')
print(f'Total após  deduplicação             : {n_apos_dedup:,}')
print(f'Redução total                        : {n_antes_dedup - n_apos_dedup:,}')

In [ ]:
# --- SEÇÃO 7: Gerar V3 FULL ---
print(SEP)
print('SEÇÃO 7 — GERANDO V3 FULL')
print(SEP)

TS = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')

df_v3_full = df_dedup[COLUNAS_V3].copy()
df_v3_full['label'] = df_v3_full['label'].astype(int)

ARQUIVO_V3_FULL = DIR_FINAL / f'dataset_final_treino_v3_full_{TS}.csv'
df_v3_full.to_csv(ARQUIVO_V3_FULL, index=False, encoding='utf-8')

print(f'V3 full salvo: {ARQUIVO_V3_FULL.name}')
print(f'Total registros: {len(df_v3_full):,}')
print()
print('Distribuição por label:')
print(df_v3_full['label'].value_counts().rename({0: 'label=0 (fake)', 1: 'label=1 (real)'}).to_string())
print()
print('Distribuição por dataset_origem:')
print(df_v3_full['dataset_origem'].value_counts().to_string())

In [ ]:
# --- SEÇÃO 8: Gerar V3 BALANCED ---
print(SEP)
print('SEÇÃO 8 — GERANDO V3 BALANCED')
print(SEP)

# Contar por label
contagem_labels = df_v3_full['label'].value_counts()
n_por_label_full = int(contagem_labels.min())
print(f'label=0: {contagem_labels.get(0, 0):,} | label=1: {contagem_labels.get(1, 0):,}')
print(f'Limite de balanceamento (min): {n_por_label_full:,} por label')
print()

# Calcular contribuição máxima por dataset_origem para evitar domínio
# Estratégia: dentro de cada label, limitar cada fonte a no máximo
# ceil(n_por_label / n_fontes_com_essa_label) * 1.5, e depois truncar ao total disponível

partes_balanced = []

for lbl in [0, 1]:
    df_lbl = df_v3_full[df_v3_full['label'] == lbl].copy()
    
    fontes = df_lbl['dataset_origem'].unique()
    n_fontes = len(fontes)
    
    # Teto por fonte: 70% do total do label para evitar domínio extremo
    teto_por_fonte = max(1, int(n_por_label_full * 0.70))
    
    partes_lbl = []
    for fonte in sorted(fontes, key=lambda x: PRIORIDADE.get(x, 99)):
        df_fonte = df_lbl[df_lbl['dataset_origem'] == fonte]
        n_amostrar = min(len(df_fonte), teto_por_fonte)
        amostra = df_fonte.sample(n=n_amostrar, random_state=SEED)
        partes_lbl.append(amostra)
    
    df_lbl_merged = pd.concat(partes_lbl, ignore_index=True)
    
    # Truncar ao limite do label
    if len(df_lbl_merged) > n_por_label_full:
        df_lbl_merged = df_lbl_merged.sample(n=n_por_label_full, random_state=SEED)
    
    partes_balanced.append(df_lbl_merged)
    print(f'label={lbl}: {len(df_lbl_merged):,} registros')
    print(df_lbl_merged['dataset_origem'].value_counts().to_string())
    print()

df_v3_balanced = pd.concat(partes_balanced, ignore_index=True)
df_v3_balanced = df_v3_balanced.sample(frac=1, random_state=SEED).reset_index(drop=True)

ARQUIVO_V3_BALANCED = DIR_FINAL / f'dataset_final_treino_v3_balanced_{TS}.csv'
df_v3_balanced.to_csv(ARQUIVO_V3_BALANCED, index=False, encoding='utf-8')

print(f'V3 balanced salvo: {ARQUIVO_V3_BALANCED.name}')
print(f'Total registros: {len(df_v3_balanced):,}')
print()
print('Distribuição por label (balanced):')
print(df_v3_balanced['label'].value_counts().rename({0: 'label=0 (fake)', 1: 'label=1 (real)'}).to_string())

In [ ]:
# --- SEÇÃO 9: Gerar V3 TEXT_CONTROL ---
print(SEP)
print('SEÇÃO 9 — GERANDO V3 TEXT_CONTROL')
print(SEP)
print('Estratégia: balancear labels DENTRO de cada faixa_tamanho_modelo')
print('Sem oversampling. Pode perder volume — dataset de ablação.\n')

faixas_ordenadas = ['curto', 'medio', 'longo', 'muito_longo']

partes_tc = []
relatorio_tc = []

for faixa in faixas_ordenadas:
    df_faixa = df_v3_full[df_v3_full['faixa_tamanho_modelo'] == faixa].copy()
    if df_faixa.empty:
        relatorio_tc.append({'faixa': faixa, 'n_label0': 0, 'n_label1': 0, 'n_selecionado': 0})
        continue
    
    n0 = int((df_faixa['label'] == 0).sum())
    n1 = int((df_faixa['label'] == 1).sum())
    n_min_faixa = min(n0, n1)
    
    relatorio_tc.append({'faixa': faixa, 'n_label0': n0, 'n_label1': n1, 'n_selecionado': n_min_faixa * 2})
    
    if n_min_faixa == 0:
        print(f'  AVISO: faixa "{faixa}" tem classe ausente — ignorada')
        continue
    
    for lbl in [0, 1]:
        parte = df_faixa[df_faixa['label'] == lbl].sample(n=n_min_faixa, random_state=SEED)
        partes_tc.append(parte)
    
    print(f'  faixa={faixa}: n0={n0} | n1={n1} → selecionado {n_min_faixa} por label')

df_relatorio_tc = pd.DataFrame(relatorio_tc)
print()
print('Resumo por faixa:')
print(df_relatorio_tc.to_string(index=False))

if partes_tc:
    df_v3_tc = pd.concat(partes_tc, ignore_index=True)
    df_v3_tc = df_v3_tc.sample(frac=1, random_state=SEED).reset_index(drop=True)
else:
    df_v3_tc = pd.DataFrame(columns=COLUNAS_V3)

ARQUIVO_V3_TC = DIR_FINAL / f'dataset_final_treino_v3_text_control_{TS}.csv'
df_v3_tc.to_csv(ARQUIVO_V3_TC, index=False, encoding='utf-8')

print()
print(f'V3 text_control salvo: {ARQUIVO_V3_TC.name}')
print(f'Total registros: {len(df_v3_tc):,}')
if len(df_v3_tc) > 0:
    print('Distribuição por label (text_control):')
    print(df_v3_tc['label'].value_counts().rename({0: 'label=0 (fake)', 1: 'label=1 (real)'}).to_string())

# Aviso metodológico
n_full = len(df_v3_full)
n_tc   = len(df_v3_tc)
pct_retido = 100.0 * n_tc / n_full if n_full > 0 else 0
print(f'\nVolume retido vs V3_full: {n_tc:,}/{n_full:,} = {pct_retido:.1f}%')
if pct_retido < 40:
    print('AVISO: volume retido < 40% — usar este dataset apenas como ablação de viés de tamanho')

In [ ]:
# --- SEÇÃO 10: Relatório Final ---
print(SEP)
print('RELATÓRIO FINAL — dataset_final_treino_v3')
print(SEP)

print()
print('[ ARQUIVOS USADOS ]')
print(f'  V2 balanced          : {ARQUIVO_V2_BALANCED.name}')
print(f'  FakeTrueBR curated   : {ARQUIVO_FAKETRUEBR.name}')
print(f'  FakeBR text_norm     : {ARQUIVO_FAKEBR_NORM.name}')
print(f'  Overlap report       : {ARQUIVO_OVERLAP_RPT.name if ARQUIVO_OVERLAP_RPT else "N/A"}')
print(f'  Near-duplicates      : {ARQUIVO_NEAR_DUPS.name if ARQUIVO_NEAR_DUPS else "N/A"}')

print()
print('[ CONTAGENS POR ETAPA ]')
print(f'  Inicial V2_PROPRIA             : {n_inicial_v2:,}')
print(f'  Inicial FAKETRUEBR             : {n_inicial_ftbr:,}')
print(f'  Inicial FAKEBR_TEXT_NORMALIZED : {n_inicial_fbr:,}')
print(f'  Total inicial                  : {n_inicial_total:,}')
print(f'  Após limpeza básica            : {n_antes_dedup:,}  (removidos: {n_sem_tpm + n_sem_label})')
print(f'  Removidos sem texto_modelo     : {n_sem_tpm}')
print(f'  Removidos sem label válido     : {n_sem_label}')
print(f'  Removidos por texto duplicado  : {n_dup_texto}')
print(f'  Removidos por URL duplicada    : {n_dup_url}')
print(f'  Removidos por near-duplicate   : {n_near_dup_removidos}')
print(f'  Removidos por conflito label   : {n_conflito}')
print(f'  Total V3 full                  : {len(df_v3_full):,}')

print()
print('[ DISTRIBUIÇÃO POR LABEL — V3 FULL ]')
vc_full = df_v3_full['label'].value_counts().sort_index()
for lbl, cnt in vc_full.items():
    nome = 'fake/negativo' if lbl == 0 else 'real/positivo'
    pct = 100.0 * cnt / len(df_v3_full)
    print(f'  label={lbl} ({nome}): {cnt:,}  ({pct:.1f}%)')

print()
print('[ DISTRIBUIÇÃO POR LABEL — V3 BALANCED ]')
vc_bal = df_v3_balanced['label'].value_counts().sort_index()
for lbl, cnt in vc_bal.items():
    nome = 'fake/negativo' if lbl == 0 else 'real/positivo'
    pct = 100.0 * cnt / len(df_v3_balanced)
    print(f'  label={lbl} ({nome}): {cnt:,}  ({pct:.1f}%)')

print()
print('[ DISTRIBUIÇÃO POR LABEL — V3 TEXT_CONTROL ]')
if len(df_v3_tc) > 0:
    vc_tc = df_v3_tc['label'].value_counts().sort_index()
    for lbl, cnt in vc_tc.items():
        nome = 'fake/negativo' if lbl == 0 else 'real/positivo'
        pct = 100.0 * cnt / len(df_v3_tc)
        print(f'  label={lbl} ({nome}): {cnt:,}  ({pct:.1f}%)')
else:
    print('  Dataset vazio.')

print()
print('[ DISTRIBUIÇÃO POR DATASET_ORIGEM — V3 FULL ]')
vc_origem = df_v3_full['dataset_origem'].value_counts()
for origem, cnt in vc_origem.items():
    pct = 100.0 * cnt / len(df_v3_full)
    print(f'  {origem:30s}: {cnt:,}  ({pct:.1f}%)')

print()
print('[ DISTRIBUIÇÃO POR DATASET_ORIGEM — V3 BALANCED ]')
vc_orig_bal = df_v3_balanced['dataset_origem'].value_counts()
for origem, cnt in vc_orig_bal.items():
    pct = 100.0 * cnt / len(df_v3_balanced)
    print(f'  {origem:30s}: {cnt:,}  ({pct:.1f}%)')

print()
print('[ DISTRIBUIÇÃO POR ORIGEM_QUALIDADE — V3 FULL ]')
print(df_v3_full['origem_qualidade'].value_counts().to_string())

print()
print('[ ESTATÍSTICAS DE TAMANHO_CHARS_MODELO — V3 FULL ]')
for lbl in [0, 1]:
    nome = 'fake' if lbl == 0 else 'real'
    s = df_v3_full[df_v3_full['label'] == lbl]['tamanho_chars_modelo']
    print(f'  label={lbl} ({nome}): média={s.mean():.0f} | mediana={s.median():.0f} | min={s.min()} | max={s.max():,}')

med0 = df_v3_full[df_v3_full['label'] == 0]['tamanho_chars_modelo'].median()
med1 = df_v3_full[df_v3_full['label'] == 1]['tamanho_chars_modelo'].median()
ratio = med1 / med0 if med0 > 0 else float('inf')
print(f'  Ratio mediana real/fake: {ratio:.2f}x')

print()
print('[ ESTATÍSTICAS DE TAMANHO_CHARS_MODELO — V3 BALANCED ]')
for lbl in [0, 1]:
    nome = 'fake' if lbl == 0 else 'real'
    s = df_v3_balanced[df_v3_balanced['label'] == lbl]['tamanho_chars_modelo']
    print(f'  label={lbl} ({nome}): média={s.mean():.0f} | mediana={s.median():.0f} | min={s.min()} | max={s.max():,}')

print()
print('[ DISTRIBUIÇÃO POR FAIXA_TAMANHO_MODELO — V3 FULL ]')
for lbl in [0, 1]:
    nome = 'fake' if lbl == 0 else 'real'
    vc = df_v3_full[df_v3_full['label'] == lbl]['faixa_tamanho_modelo'].value_counts()
    print(f'  label={lbl} ({nome}): {dict(vc)}')

print()
print('[ ARQUIVOS SALVOS ]')
print(f'  V3 full          : {ARQUIVO_V3_FULL.name}')
print(f'  V3 balanced      : {ARQUIVO_V3_BALANCED.name}')
print(f'  V3 text_control  : {ARQUIVO_V3_TC.name}')

print()
print('[ RECOMENDAÇÃO ]')
if ratio <= 1.5:
    print('  Ratio mediana real/fake ≤ 1.5 → viés de tamanho moderado.')
    print('  → RECOMENDAÇÃO: usar V3_BALANCED como dataset principal de treino.')
    print('  → V3_TEXT_CONTROL como experimento de ablação secundário.')
else:
    print(f'  ATENÇÃO: Ratio mediana real/fake = {ratio:.2f}x → viés de tamanho significativo.')
    print('  → RECOMENDAÇÃO: usar V3_TEXT_CONTROL para treino inicial (viés controlado).')
    print('  → Usar V3_BALANCED com class_weight para compensar desbalanceamento de tamanho.')

In [ ]:
# --- Salvar log de conflitos (se houver) ---
if len(df_conflitos_log) > 0:
    ARQUIVO_CONFLITOS = DIR_FINAL / f'dataset_v3_conflitos_removidos_{TS}.csv'
    df_conflitos_log.to_csv(ARQUIVO_CONFLITOS, index=False, encoding='utf-8')
    print(f'AVISO: {len(df_conflitos_log)} registros com conflito de label salvos em:')
    print(f'  {ARQUIVO_CONFLITOS.name}')
else:
    print('Nenhum conflito de label detectado — nenhum arquivo de log de conflitos gerado.')

## Conclusão Metodológica

### dataset_final_treino_v3 — Composição e Uso

**Composição do V3:**
- A V3 une os dados próprios do projeto (`V2_PROPRIA`) com dois datasets acadêmicos referenciados (`FakeTrueBR` e `Fake.Br`).
- A **V2** representa o trabalho original do projeto: coleta via Google Fact Check (`GFC`) e RSS de notícias reais, com curadoria manual e rotulagem forte (`ROTULO_FORTE`). Ela é preservada como bloco base oficial e não é reprocessada.
- **FakeTrueBR** e **Fake.Br** entram com `origem_qualidade = ROTULO_ACADEMICO`, indicando que os rótulos foram atribuídos por curadoria acadêmica, não pelo pipeline próprio.

**Tratamento do Fake.Br:**
- O Fake.Br usa `texto_principal_modelo` (texto truncado/normalizado) para reduzir o viés de tamanho textual que o dataset original introduz — artigos reais tendem a ser muito mais longos que notícias falsas.
- O campo `texto_principal` original é preservado para fins de auditoria e rastreabilidade.
- Em treino, o campo preferencial é sempre `texto_principal_modelo`.

**Prioridade de deduplicação:**
1. `V2_PROPRIA` (maior confiança de rotulagem)
2. `FAKEBR_TEXT_NORMALIZED` (texto normalizado por tamanho, dataset maior)
3. `FAKETRUEBR` (removido em caso de near-duplicate com FakeBR)

**Uso de cada variante:**
- `V3_FULL`: análise exploratória geral, experimentos com `class_weight`, baseline de cobertura máxima.
- `V3_BALANCED`: **candidato principal para treino V3** — volume razoável, sem oversampling, distribuição equilibrada por fonte.
- `V3_TEXT_CONTROL`: experimento de ablação para isolar viés de tamanho textual — usar apenas se o ratio mediana real/fake no V3_full indicar viés expressivo (> 1.5×).

**O que NÃO foi feito nesta fase:**
- Nenhum modelo foi treinado.
- Nenhum arquivo existente foi alterado (V1, V2, curated originais).
- Nenhum arquivo `.joblib` foi criado.
- Nenhum CSV existente foi sobrescrito.